In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import pandas as pd
df =pd.read_csv("/content/drive/MyDrive/skin-disease-detection-cnn/data/HAM10000_metadata")

In [23]:
import os
part1_path = "/content/drive/MyDrive/skin-disease-detection-cnn/data/HAM10000_images_part_1"
part2_path = "/content/drive/MyDrive/skin-disease-detection-cnn/data/HAM10000_images_part_2"

In [24]:
def get_image_path(image_id):
    filename = image_id + ".jpg"
    path1 = os.path.join(part1_path, filename)
    if os.path.exists(path1):
        return path1
    else:
        return os.path.join(part2_path, filename)

In [25]:
unique_lesions = df['lesion_id'].unique()
unique_lesions
print(len(unique_lesions))

7470


In [26]:
lesion_labels = df.groupby('lesion_id')['dx'].first()
print(lesion_labels.head())
from sklearn.model_selection import train_test_split

train_lesions, temp_lesions = train_test_split(
    lesion_labels.index,
    test_size=0.30,
    stratify=lesion_labels.values,
    random_state=42
)

val_lesions, test_lesions = train_test_split(
    temp_lesions,
    test_size=0.50,
    random_state=42
)

print(len(train_lesions))
print(len(val_lesions))
print(len(test_lesions))

lesion_id
HAM_0000000     nv
HAM_0000001    bkl
HAM_0000002    mel
HAM_0000003     nv
HAM_0000004     nv
Name: dx, dtype: object
5229
1120
1121


In [27]:
# Train dataframe
train_df = df[df['lesion_id'].isin(train_lesions)].copy()

# Validation dataframe
val_df = df[df['lesion_id'].isin(val_lesions)].copy()

# Test dataframe
test_df = df[df['lesion_id'].isin(test_lesions)].copy()

# Shapes print
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# Train class distribution
print(train_df['dx'].value_counts(normalize=True))

# Validation class distribution
print(val_df['dx'].value_counts(normalize=True))

# Test class distribution
print(test_df['dx'].value_counts(normalize=True))

Train shape: (6981, 8)
Validation shape: (1518, 8)
Test shape: (1516, 8)
dx
nv       0.670821
mel      0.110729
bkl      0.110586
bcc      0.051712
akiec    0.031801
vasc     0.014181
df       0.010170
Name: proportion, dtype: float64
dx
nv       0.681159
bkl      0.102767
mel      0.099473
bcc      0.056653
akiec    0.031621
df       0.016469
vasc     0.011858
Name: proportion, dtype: float64
dx
nv       0.651715
mel      0.124670
bkl      0.112797
bcc      0.044195
akiec    0.037599
vasc     0.016491
df       0.012533
Name: proportion, dtype: float64


In [28]:
# Import LabelEncoder for converting text labels into numeric labels
from sklearn.preprocessing import LabelEncoder

# Create the encoder object
encoder = LabelEncoder()

# Fit the encoder only on training labels
# This allows the model to learn all unique classes from training data
encoder.fit(train_df['dx'])

# Convert text labels into numeric labels for training data
train_df['label'] = encoder.transform(train_df['dx'])

# Apply the same learned mapping on validation data
val_df['label'] = encoder.transform(val_df['dx'])

# Apply the same learned mapping on test data
test_df['label'] = encoder.transform(test_df['dx'])

# Print all classes learned by the encoder
print(encoder.classes_)

['akiec' 'bcc' 'bkl' 'df' 'mel' 'nv' 'vasc']


In [29]:
# Create image path column for training data
train_df['image_path'] = train_df['image_id'].apply(get_image_path)

# Create image path column for validation data
val_df['image_path'] = val_df['image_id'].apply(get_image_path)

# Create image path column for test data
test_df['image_path'] = test_df['image_id'].apply(get_image_path)

# Display a few sample paths
train_df[['image_id', 'image_path']].head()

,image_id,image_path
0,ISIC_0027419,/content/drive/MyDrive/skin-disease-detection-...
1,ISIC_0025030,/content/drive/MyDrive/skin-disease-detection-...
2,ISIC_0026769,/content/drive/MyDrive/skin-disease-detection-...
3,ISIC_0025661,/content/drive/MyDrive/skin-disease-detection-...
4,ISIC_0031633,/content/drive/MyDrive/skin-disease-detection-...


For Additional Verification,we'll do this

In [30]:
import os
sample_path = train_df['image_path'].iloc[0]
print(sample_path)
print(os.path.exists(sample_path))

/content/drive/MyDrive/skin-disease-detection-cnn/data/HAM10000_images_part_1/ISIC_0027419.jpg
True


In [31]:
from PIL import Image
import numpy as np

# Take first image path from training dataframe
img_path = train_df.iloc[0]['image_path']

# Load image
image = Image.open(img_path)

# Resize image
image = image.resize((224, 224))

# Convert image into NumPy array
image = np.array(image)

# Normalize pixel values
image = image / 255.0

# Check final shape
print(image.shape)

# Check maximum pixel value
print(image.max())

(224, 224, 3)
1.0


In [32]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=20,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# Validation and test generator (normalization only)
val_test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [33]:
# Train generator
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='image_path',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='raw'
)

# Validation generator
val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='image_path',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='raw'
)

Found 6981 validated image filenames.
Found 1518 validated image filenames.


In [34]:
test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='image_path',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='raw',
    shuffle=False
)

Found 1516 validated image filenames.


Test generator only rescales images without augmentation and keeps shuffle=False to ensure fair and ordered evaluation of unseen data

# Phase 2 — Preprocessing & Augmentation

## What Was Done
- Mapped each image to its correct folder (part 1 or part 2)
- Split data 70/15/15 on lesion_id to avoid data leakage
- Encoded text labels to numbers using LabelEncoder
- Images resized to 224x224 and normalized to 0-1 range
- Set up batch generators for memory-efficient loading

## Key Decisions
- **Lesion-based split:** Same lesion shouldn't appear in both
  train and test — that would be cheating
- **Stratified split:** All 7 classes stay proportional in
  every split — important because dataset is imbalanced
- **Augmentation on train only:** Val and test should reflect
  real-world conditions — no artificial changes
- **LabelEncoder fit on train only:** Standard ML practice —
  test data should never influence any step of training

## Class Imbalance Note
nv dominates at 67% — remaining 6 classes share 33% only.
This will be handled in Phase 3 using class weights.

In [39]:
import os

preprocessed_path = "/content/drive/MyDrive/skin-disease-detection-cnn/data/preprocessed"
os.makedirs(preprocessed_path, exist_ok=True)
print("Folder created!")




Folder created!


In [38]:
# Save preprocessed DataFrames as CSV files
# index=False is used so that extra index column is not saved


train_df.to_csv(preprocessed_path + "/train.csv", index=False)
val_df.to_csv(preprocessed_path + "/val.csv", index=False)
test_df.to_csv(preprocessed_path + "/test.csv", index=False)

print("All CSVs saved!")

# Verify
import os
print(os.listdir(preprocessed_path))

All CSVs saved!
['train.csv', 'val.csv', 'test.csv']
